In [ ]:

from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor
from scripts.metrics import StatsCollector, StatsHandler


from sampo.base import SAMPO
from sampo.scheduler.genetic import GeneticScheduler
from sampo.schemas.graph import WorkGraph



import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict

# Genetic Algrotihm basic SAMPO  vs Genetic Algorithm with LLMs generated init population

Цель исследования:    
    - Проверить сходимость за сколько был достигнут GAP = 0,   
    Гипотеза: Iter_LLM <  Iter_GA    
    - Оптимальность  
    Гипотеза: makespan_LLM < makespan_GA


In [ ]:
def init_experiment(GA_params, model_names, imprortance):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for model_name in model_names:
        model = LLMHeuristicScheduler(model_name, **GA_params, 
                                      imprortance=imprortance)
        solvers_dict[model_name] = model
    return solvers_dict

def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    return df_res

def run_experiment(solvers_dict, wg, contractors, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    experiment_logs = defaultdict(list)
    for i in range(N_runs):
        for solver, model in solvers_dict.items():
            model.schedule(wg, contractors)
            experiment_logs[solver].append(sc.items)
            sc.clear()
    return experiment_logs



GA  = {
    'number_of_generation' : 3,
    'size_of_population' : 60,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 


wg , contractors  = WorkGraph.loadf('wgs/100', '100_0'), contractor(N=5)


#wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_6'), contractor(N=5)
solvers_dict = init_experiment(GA, ['claude_haiku_4.5', ], 11)
experiment_logs = run_experiment(solvers_dict, wg, contractors, 3)


In [ ]:
sns.lineplot(experiment_df(experiment_logs),
              x='Time', y='Makespan', hue='Algorithm', 
              units='run', estimator=None, alpha=0.7)

plt.ylim((25, 150))
plt.xlim((0, 75))

# Проверка гипотезы о весах в ГА

In [ ]:
def tail_share(weights, tail_indices, p_init=1/3):
    W = sum(weights)
    W_tail = sum(weights[tail_indices:])
    return p_init * W_tail / W  # доля в общей популяции


print('Deepseek Chat')
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 9, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [2] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [1] * 9, tail_indices=7) / 0.33 )
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [6] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [5] * 9, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [12] * 9, tail_indices=7), tail_share([2, 2, 2, 2, 1, 1, 1] + [10] * 9, tail_indices=7) / 0.33)


print('Reasoner')
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [0.5] * 5, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [2] * 5, tail_indices=7) / 0.33 )
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [6] * 5, tail_indices=7) / 0.33)
print( tail_share([2, 2, 2, 2, 1, 1, 1] + [12] * 5, tail_indices=7) )


In [ ]:
from sampo.schemas.graph import WorkGraph

def init_experiment(GA_params, model_names, imprt):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for model_name in model_names:
        model = LLMHeuristicScheduler(model_name, **GA_params, imprt=imprt)
        solvers_dict[model_name] = model
    return solvers_dict

def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        if name == 'weight_param':
            continue
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    df_res['weight_param'] = experiment_logs['weight_param'] 
    return df_res

def run_experiments(GA, wg, weights, contractors, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    res_dfs = []
    for weight in weights:
        experiment_logs = defaultdict(list)
        solvers_dict = init_experiment(GA, ['deepseek_chat', 'deepseek_reasoner'], imprt=weight)
        for _ in range(N_runs):
            for solver, model in solvers_dict.items():
                if weight >= weights[0] and solver == 'genetic':
                # Skip other experiments
                    continue
                model.schedule(wg, contractors)
                experiment_logs[solver].append(sc.items)
                experiment_logs['weight_param'] = weight
                sc.clear()
            res_dfs.append(experiment_df(experiment_logs))
    
    return res_dfs


GA  = {
    'number_of_generation' : 150,
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 


weights = (8, 9, 10)


wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_9'), contractor(N=5)
dfs = run_experiments(GA, wg, contractors, 5)


In [ ]:
data = pd.concat(dfs)
data.to_csv('weight_experiment2_df.csv', index=False)

#data.to_csv('weight_experiment2_weights_df.csv', index=False)


# Meta

# GA  = {
#     'number_of_generation' : 150,
#     'size_of_population' : 50,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 


#df_experiment.to_csv('weight_experiment_df.csv', index=False)


# Meta

# GA  = {
#     'number_of_generation' : 100,
#     'size_of_population' : 50,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 



In [ ]:
# from sampo.schemas.graph import WorkGraph
# import os
# from sampo_api import contractor

# maxx, minn = -float('inf'), float('inf')
# for f in os.listdir('wgs/small_synth'):
#         wg , contractors = WorkGraph.loadf('wgs/small_synth', f[:-5]), contractor(N=5)
#         N = len(wg.nodes) 
#         print( f, N)
#         maxx = max(maxx, N)
#         minn = min(minn, N)

# print(maxx, minn)

# Experiment with structure in GA
    - `Classic` GA vs `High Weight` in init_popul vs `Only model generated` init_popul

In [ ]:
from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor
from sampo.schemas.graph import WorkGraph

wg , contractors  = WorkGraph.loadf('wgs/100', '100_0'), contractor(N=10)


def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        for i, run in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 
                               'run' : i, 'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    return df_res



def init_experiment(GA_params, 
                    model_name, 
                    structures = (None, 'onlyGeneratedHeurisitcs'), 
                    imprt = 10):
    solvers_dict = {}
    solvers_dict['genetic'] = GeneticScheduler(**GA_params)
    for structure in structures:
        model = LLMHeuristicScheduler(model_name, 
                                      **GA_params, 
                                      type_init_pop_structure=structure, 
                                      imprortance=imprt)
        if structure is None:
            structure = 'highWeights'
        solvers_dict[model_name + '_' + structure] = model
    return solvers_dict

def run_experiments(GA, wg, contractors, model_name, N_runs):
    sc = StatsCollector()
    stats_handler = StatsHandler(sc)
    SAMPO.logger.addHandler(stats_handler)
    res_dfs = []
    experiment_logs = defaultdict(list)
    solvers_dict = init_experiment(GA, model_name)
    for _ in range(N_runs):
        for solver, model in solvers_dict.items():
            model.schedule(wg, contractors)
            experiment_logs[solver].append(sc.items)
            sc.clear()
            res_dfs.append(experiment_df(experiment_logs))
    return res_dfs


GA  = {
    'number_of_generation' : 100,
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 

dfs = run_experiments(GA, wg, contractors, 'deepseek_reasoner', 4)

In [ ]:
pd.concat(dfs).groupby('Algorithm').Makespan.min()

In [ ]:
pd.concat(dfs).to_csv('structure_experiment_DRChat.csv', index=False)

In [ ]:
str(None)

In [ ]:
# from sampo.schemas.graph import WorkGraph
# WorkGraph.loadf('wgs/100', '100_0')

In [ ]:
GA  = {
    'number_of_generation' : 150,
    'size_of_population' : 50,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 
scheduler = LLMHeuristicScheduler('deepseek_reasoner', 
                                  **GA,  
                                 type_init_pop_structure='onlyGeneratedHeurisitcs')
scheduler.schedule(wg, contractors)


# Эксперимент со структурой популяции


In [ ]:

from LLMHeuristicScheduler.base import LLMHeuristicScheduler
from sampo_api import contractor
from scripts.metrics import StatsCollector, StatsHandler


from sampo.base import SAMPO
from sampo.scheduler.genetic import GeneticScheduler

from sampo.schemas.graph import WorkGraph



import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict



In [ ]:
# from LLMHeuristicScheduler.base import LLMHeuristicScheduler
# from sampo_api import contractor
# from sampo.schemas.graph import WorkGraph

# wg , contractors  = WorkGraph.loadf('wgs/small_synth', 'wg_32'), contractor(N=5)


# def experiment_df(experiment_logs):
#     res = []
#     for name, runs in experiment_logs.items():
#         for i, run in enumerate(runs, start=1):
#             df = pd.DataFrame({'Makespan' : run, 'Time' : range(1, len(run) + 1), 
#                                'run' : i, 'Algorithm' : name})
#             res.append(df)
#     df_res = pd.concat(res)
#     return df_res



# def init_experiment(GA_params, 
#                     model_name, 
#                     structures = ('switch_Topological', 
#                                   'switch_ALL'), 
#                     imprt = 10):
#     solvers_dict = {}
#     solvers_dict['genetic'] = GeneticScheduler(**GA_params)
#     for structure in structures:
#         print(f'Init {structure} for GA')
#         model = LLMHeuristicScheduler(model_name, 
#                                       **GA_params, 
#                                       type_init_pop_structure=structure, 
#                                       imprortance=imprt)
#         # if structure is None:
#         #     structure = 'x'
#         solvers_dict[model_name + '_' + structure] = model
#     return solvers_dict


# def run_experiments(GA, wg, contractors, model_name, N_runs):
#     sc = StatsCollector()
#     stats_handler = StatsHandler(sc)
#     SAMPO.logger.addHandler(stats_handler)
#     res_dfs = []
#     experiment_logs = defaultdict(list)
#     solvers_dict = init_experiment(GA, model_name)
#     for _ in range(N_runs):
#         for solver, model in solvers_dict.items():
#             model.schedule(wg, contractors)
#             experiment_logs[solver].append(sc.items)
#             sc.clear()
#     res_dfs.append(experiment_df(experiment_logs))
#     return res_dfs

# # 72
# GA  = {
#     'number_of_generation' : 1,  
#     'size_of_population' : 60,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 

# dfs = run_experiments(GA, wg, contractors, 'combined_models', 2)

In [ ]:
# pd.concat(dfs)

# Параллельный эскперимент 

In [1]:
from concurrent.futures import ProcessPoolExecutor
from Experiments.structure import run_one_experiment
from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict
import pandas as pd


def experiment_df(experiment_logs):
    res = []
    for name, runs in experiment_logs.items():
        for i, makespan_series in enumerate(runs, start=1):
            df = pd.DataFrame({'Makespan' : makespan_series, 
                               'Time' : range(1, len(makespan_series) + 1), 
                               'run' : i, 
                               'Algorithm' : name})
            res.append(df)
    df_res = pd.concat(res)
    return df_res


def run_experiments_parallel(GA,
                             size,
                             n_contractors, 
                             model_name, 
                             N_runs, 
                             structures = ('switch_Topological',
                                           'switch_ALL',
                                           'switch_LFTrand',
                                           'switch_HEFT',
                                            None),
                             max_workers = None):

    # size = 30, 50, 100
    # GA, size, n, structures, model_name = args
    args = [(GA, size, n_contractors, structures, model_name) for _ in range(N_runs)]

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(run_one_experiment, args))

    experiment_logs = defaultdict(list)
    
    for run_logs in results:
        for solver, logs in run_logs.items():
            experiment_logs[solver].extend(logs)

    return [experiment_df(experiment_logs)]


GA  = {
    'number_of_generation' : 1,  
    'size_of_population' : 100,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 

# 91 was

# res_j30_dfs = run_experiments_parallel(GA, 
#                                        30,
#                                        2,
#                                       'combined_models', 
#                                        N_runs=30, 
#                                        max_workers=6)

# pd.concat(res_j30_dfs).to_csv('Experiments/Runs/final_j30.csv')

# res_j50_dfs = run_experiments_parallel(GA, 
#                                        50,
#                                        3,
#                                       'combined_models', 
#                                        N_runs=30, 
#                                        max_workers=6)
# pd.concat(res_j50_dfs).to_csv('Experiments/Runs/final_j50.csv')

res_j100_dfs = run_experiments_parallel(GA, 
                                       100,
                                       3,
                                      'combined_models', 
                                       N_runs=1, 
                                       max_workers=None)

pd.concat(res_j100_dfs).to_csv('Experiments/Runs/new_3final_j100.csv')


Can not find native module; switching to default
Can not find native module; switching to default
Init switch_Topological for GA
Init switch_ALL for GA
Init switch_LFTrand for GA
Init switch_HEFT for GA
Init None for GA
Genetic optimizing took 16.376733779907227 ms


[SAMPO] [INFO] Toolbox initialization & first population took 16.442060470581055 ms
[SAMPO] [INFO] First population evaluation took 2705.559015274048 ms
[SAMPO] [INFO] -- Generation 1, population=100, best fitness=(246.0,) --
[SAMPO] [INFO] Final fitness: (246.0,)
[SAMPO] [INFO] Generations processing took 2775.1359939575195 ms
[SAMPO] [INFO] Full genetic processing took 9094.113111495972 ms
[SAMPO] [INFO] Evaluation time: 5399.008274078369


243 26_heuristic
243 32_heuristic
494 13_heuristic
211 07_heuristic
232 18_heuristic
202 39_heuristic
239 15_heuristic
191 20_heuristic
451 34_heuristic
203 33_heuristic
203 27_heuristic
188 12_heuristic
281 19_heuristic
228 38_heuristic
203 14_heuristic
287 35_heuristic
239 21_heuristic
212 08_heuristic
255 29_heuristic
233 11_heuristic
240 05_heuristic
494 24_heuristic
257 30_heuristic
203 22_heuristic
217 36_heuristic
247 17_heuristic
208 03_heuristic
200 09_heuristic
638 28_heuristic
494 04_heuristic
206 10_heuristic
247 31_heuristic
215 25_heuristic
256 40_heuristic
216 37_heuristic
214 23_heuristic
243 02_heuristic
231 16_heuristic
38
Genetic optimizing took 99.18498992919922 ms


[SAMPO] [INFO] Toolbox initialization & first population took 99.65205192565918 ms
[SAMPO] [INFO] First population evaluation took 2884.719133377075 ms
[SAMPO] [INFO] -- Generation 1, population=100, best fitness=(188.0,) --
[SAMPO] [INFO] Final fitness: (188.0,)
[SAMPO] [INFO] Generations processing took 2821.7709064483643 ms
[SAMPO] [INFO] Full genetic processing took 5809.300184249878 ms
[SAMPO] [INFO] Evaluation time: 5594.770193099976


243 26_heuristic
243 32_heuristic
494 13_heuristic


KeyboardInterrupt: 

In [ ]:
pd.concat(res_j50_dfs)

In [ ]:
res_j30_dfs

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
_, axes = plt.subplots(*(1,2), figsize=(12,8), sharey=True)


axes = axes.flatten()

sns.lineplot( data = pd.concat(res_j30_dfs),
              x='Time', y='Makespan', hue='Algorithm', 
              units='run',
              estimator=None, alpha=0.8, ax = axes[0])


sns.lineplot( data = pd.concat(res_j30_dfs),
              x='Time', y='Makespan', hue='Algorithm', 
              estimator='mean', alpha=0.8, ax = axes[1])

In [ ]:
pd.concat(res_j30_dfs).query('Time == 12').groupby('Algorithm').Makespan.describe()

In [ ]:
pd.concat(res_j30_dfs).query('Time == 3').groupby('Algorithm').Makespan.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
_, axes = plt.subplots(*(1,2), figsize=(12,8), sharey=True)


axes = axes.flatten()

sns.lineplot( data = pd.read_csv('Experiments/Runs/3_res_j30_dfs.csv'),
              x='Time', y='Makespan', hue='Algorithm', 
              units='run',
              estimator=None, alpha=0.8, ax = axes[0])


sns.lineplot( data = pd.read_csv('Experiments/Runs/3_res_j30_dfs.csv'),
              x='Time', y='Makespan', hue='Algorithm', 
              estimator='mean', alpha=0.8, ax = axes[1])

In [ ]:
pd.read_csv('Experiments/Runs/3_res_j100_dfs.csv').query('Time == 120').groupby('Algorithm').Makespan.describe()

In [ ]:
import numpy as np

2.09 / np.sqrt(24), 2.08 / np.sqrt(24)

In [ ]:
# pd.concat(res_j30_dfs).to_csv('3_res_j30_dfs.csv', index=False)
# GA  = {
#     'number_of_generation' : 90,  
#     'size_of_population' : 81,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 
# res_j50_dfs = run_experiments_parallel(GA, 
#                                    50,
#                                    3,
#                                   'combined_models', 
#                                    N_runs=24, 
#                                    max_workers=6)
# pd.concat(res_j50_dfs).to_csv('3_res_j50_dfs.csv', index=False)
# GA  = {
#     'number_of_generation' : 120,  
#     'size_of_population' : 81,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 
# res_j100_dfs = run_experiments_parallel(GA, 
#                                    100,
#                                    5,
#                                   'combined_models', 
#                                    N_runs=24, 
#                                    max_workers=6)

# pd.concat(res_j100_dfs).to_csv('3_res_j100_dfs.csv', index=False)
# GA  = {
#     'number_of_generation' : 150,  
#     'size_of_population' : 60,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 

# res_j100_dfs = run_experiments_parallel(GA, 
#                                    100,
#                                    15,
#                                   'deepseek_reasoner', 
#                                    N_runs=20, 
#                                    max_workers=6)

# Checks

- 0 Base topology (n//2 for top)
- 1 n//4
- 2 c <<
- 3 for all 1 and 2

In [ ]:
#pd.concat(res_j30_dfs)

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


res_j30_dfs = pd.read_csv('3_res_j30_dfs.csv')

_, axes = plt.subplots(*(1,2), figsize=(12,8), sharey=True)


axes = axes.flatten()

sns.lineplot( data = res_j30_dfs,
              x='Time', y='Makespan', hue='Algorithm', 
              units='run',
              estimator=None, alpha=0.8, ax = axes[0])


sns.lineplot( data = res_j30_dfs,
              x='Time', y='Makespan', hue='Algorithm', 
              estimator='mean', alpha=0.8, ax = axes[1])




In [ ]:
res_j30_dfs.groupby('Algorithm').Makespan.min()

In [ ]:
#pd.concat(res_j30_dfs)

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


res_j30_dfs = pd.read_csv('1_res_j30_dfs.csv')

_, axes = plt.subplots(*(1,2), figsize=(12,8), sharey=True)


axes = axes.flatten()

sns.lineplot( data = res_j30_dfs,
              x='Time', y='Makespan', hue='Algorithm', 
              units='run',
              estimator=None, alpha=0.8, ax = axes[0])


sns.lineplot( data = res_j30_dfs,
              x='Time', y='Makespan', hue='Algorithm', 
              estimator='mean', alpha=0.8, ax = axes[1])




In [ ]:
# pd.concat(res_j30_dfs).groupby('Algorithm').Makespan.min()

In [ ]:
GA  = {
    'number_of_generation' : 100,  
    'size_of_population' : 81,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 

res_j50_dfs = run_experiments_parallel(GA, 
                                   50,
                                   10,
                                  'combined_models', 
                                   N_runs=20, 
                                   max_workers=5)




In [ ]:
pd.concat(res_j50_dfs).to_csv('1_res_j50_dfs.csv', index=False)

In [ ]:
#pd.concat(res_j30_dfs)

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


res_j30_dfs
_, axes = plt.subplots(*(1,2), figsize=(12,8), sharey=True)


axes = axes.flatten()

sns.lineplot( data = pd.concat(res_j50_dfs),
              x='Time', y='Makespan', hue='Algorithm', 
              units='run',
              estimator=None, alpha=0.8, ax = axes[0])


sns.lineplot( data = pd.concat(res_j50_dfs),
              x='Time', y='Makespan', hue='Algorithm', 
              estimator='mean', alpha=0.8, ax = axes[1])




In [ ]:
pd.concat(res_j50_dfs).groupby('Algorithm').Makespan.min()

In [ ]:
# GA  = {
#     'number_of_generation' : 150,  
#     'size_of_population' : 60,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 

# res_j100_dfs = run_experiments_parallel(GA, 
#                                    100,
#                                    15,
#                                   'combined_models', 
#                                    N_runs=24, 
#                                    max_workers=6)


# pd.concat(res_j100_dfs).to_csv('1_res_j100_dfs.csv', index=False)


In [ ]:
#pd.concat(res_j30_dfs)

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


_, axes = plt.subplots(*(1,2), figsize=(12,8), sharey=True)


axes = axes.flatten()

sns.lineplot( data = pd.concat(res_j100_dfs),
              x='Time', y='Makespan', hue='Algorithm', 
              units='run',
              estimator=None, alpha=0.8, ax = axes[0])


sns.lineplot( data = pd.concat(res_j100_dfs),
              x='Time', y='Makespan', hue='Algorithm', 
              estimator='mean', alpha=0.8, ax = axes[1])




In [ ]:
GA  = {
    'number_of_generation' : 120,  
    'size_of_population' : 81,
    'mutate_order' : 0.05,
    'mutate_resources': 0.05,
    'mutate_zones': 0.05} 

res_j100_dfs = run_experiments_parallel(GA, 
                                   100,
                                   5,
                                  'combined_models', 
                                   N_runs=6, 
                                   max_workers=6)


pd.concat(res_j100_dfs).to_csv('2_res_j100_dfs.csv', index=False)


In [ ]:
from sampo.schemas.graph import WorkGraph
from sampo_api import contractor
from sampo.scheduler.genetic import GeneticScheduler
wg, contractors =  WorkGraph.loadf('wgs/100', '100_0'), contractor(N=4)
GeneticScheduler(number_of_generation=1).schedule(wg,contractors)

In [ ]:
pd.concat(res_j100_dfs).groupby('Algorithm').Makespan.min()

In [ ]:
# GA  = {
#     'number_of_generation' : 150,  
#     'size_of_population' : 60,
#     'mutate_order' : 0.05,
#     'mutate_resources': 0.05,
#     'mutate_zones': 0.05} 

# res_j100_dfs = run_experiments_parallel(GA, 
#                                    100,
#                                    15,
#                                   'deepseek_reasoner', 
#                                    N_runs=20, 
#                                    max_workers=6)

In [ ]:
# wg , contractors  = WorkGraph.loadf('wgs/100', '100_0'), contractor(N=10)


In [ ]:
# df_ = pd.concat(res_dfs)
# df_.to_csv('structure_test_DeepseekReasoner_200.csv', index = False)

# #  WorkGraph.loadf('wgs/small_synth', 'wg_32'), contractor(N=5) | Res 1, 110 times, structure_test_DeepseekChat

In [ ]:
import pandas as pd
df_ = pd.read_csv('structure_test_DeepseekReasoner_200.csv')

In [ ]:
df_.groupby('Algorithm').Makespan.min()